# 2. Private Access — Service Endpoints, Private Endpoints, Private Link

This is one of the most confusing areas on AZ-500. There are three ways to access Azure PaaS services privately, and the exam loves asking which one to use.

## The three options compared

| | Service Endpoint | Private Endpoint | Private Link Service |
|-|-----------------|-----------------|---------------------|
| **What** | Optimized route from VNet to PaaS | Private IP in your VNet for PaaS | Expose YOUR service behind a load balancer |
| **Traffic path** | Still goes to public IP (via MS backbone) | Goes to private IP in your subnet | Goes to private IP via Private Endpoint |
| **DNS** | Uses public FQDN | Custom DNS (privatelink zone) | Custom DNS |
| **Cross-VNet** | No (VNet-local only) | Yes (via VNet peering) | Yes (even cross-tenant) |
| **On-prem access** | No | Yes (via VPN/ExpressRoute) | Yes |
| **NSG support** | No (service endpoints bypass NSGs) | Yes | Yes |
| **Cost** | Free | ~$7.30/month + data | ~$7.30/month + data |
| **Use when** | Quick win, single-VNet | Zero-trust, multi-VNet, on-prem | Exposing your own service to consumers |

In [ ]:
import json

# Decision tree: which private access method to use?
def recommend_private_access(scenario: dict) -> dict:
    needs_onprem = scenario.get('on_prem_access', False)
    needs_cross_vnet = scenario.get('cross_vnet', False)
    needs_nsg = scenario.get('nsg_enforcement', False)
    is_own_service = scenario.get('own_service', False)
    budget_sensitive = scenario.get('budget_sensitive', False)
    
    if is_own_service:
        return {'recommendation': 'Private Link Service', 'reason': 'Exposing your own service to consumers via Private Endpoint'}
    if needs_onprem or needs_cross_vnet or needs_nsg:
        return {'recommendation': 'Private Endpoint', 'reason': f'Required for: {"on-prem" if needs_onprem else ""} {"cross-VNet" if needs_cross_vnet else ""} {"NSG enforcement" if needs_nsg else ""}'.strip()}
    if budget_sensitive:
        return {'recommendation': 'Service Endpoint', 'reason': 'Free, single-VNet, quick to set up'}
    return {'recommendation': 'Private Endpoint', 'reason': 'Best practice for zero-trust (default recommendation)'}

scenarios = [
    {'name': 'Storage account accessed from one VNet only', 'budget_sensitive': True},
    {'name': 'SQL Database accessed from VNet + on-prem', 'on_prem_access': True},
    {'name': 'Key Vault with NSG restrictions', 'nsg_enforcement': True},
    {'name': 'Multi-VNet hub-spoke accessing Storage', 'cross_vnet': True},
    {'name': 'SaaS vendor exposing API to customers', 'own_service': True},
    {'name': 'Zero-trust architecture for all PaaS', },
]

print('=== Private Access Decision Guide ===\n')
for s in scenarios:
    result = recommend_private_access(s)
    print(f'📋 {s["name"]}')
    print(f'   → {result["recommendation"]} — {result["reason"]}\n')

## Service Endpoints — implementation

```bash
# Enable service endpoint for Storage on a subnet
az network vnet subnet update -g rg-prod --vnet-name vnet-prod \
  -n sn-app --service-endpoints Microsoft.Storage

# Lock down storage account to only accept traffic from that subnet
az storage account network-rule add -g rg-prod -n mystorageaccount \
  --vnet-name vnet-prod --subnet sn-app

# Set default action to deny (block all except VNet rules)
az storage account update -g rg-prod -n mystorageaccount \
  --default-action Deny
```

## Private Endpoints — implementation

```bash
# Create Private Endpoint for a Storage Account
az network private-endpoint create -g rg-prod -n pe-storage \
  --vnet-name vnet-prod --subnet sn-private-endpoints \
  --private-connection-resource-id /subscriptions/.../storageAccounts/mysa \
  --group-id blob --connection-name mysa-blob-conn

# Create Private DNS Zone (so mysa.blob.core.windows.net resolves to private IP)
az network private-dns zone create -g rg-prod -n privatelink.blob.core.windows.net

# Link DNS zone to VNet
az network private-dns link vnet create -g rg-prod \
  --zone-name privatelink.blob.core.windows.net \
  --name link-vnet-prod --virtual-network vnet-prod \
  --registration-enabled false

# Create DNS record group
az network private-endpoint dns-zone-group create -g rg-prod \
  --endpoint-name pe-storage --name default \
  --private-dns-zone privatelink.blob.core.windows.net \
  --zone-name blob
```

### DNS is the hard part

After creating a Private Endpoint, `mysa.blob.core.windows.net` must resolve to the **private IP** (e.g., 10.0.4.5), not the public IP. This requires:
1. A Private DNS zone (`privatelink.blob.core.windows.net`)
2. Linked to your VNet
3. An A record pointing to the PE's private IP

If DNS isn't configured, your app still resolves the public IP and the traffic goes through the internet.

## App Service VNet Integration

App Service (and Functions) can be integrated with a VNet for **outbound** traffic:

```bash
# Integrate App Service with a VNet (outbound traffic goes through VNet)
az webapp vnet-integration add -g rg-prod -n my-webapp \
  --vnet vnet-prod --subnet sn-app-integration
```

For **inbound** private access to App Service, use a Private Endpoint:

```bash
az network private-endpoint create -g rg-prod -n pe-webapp \
  --vnet-name vnet-prod --subnet sn-private-endpoints \
  --private-connection-resource-id /subscriptions/.../sites/my-webapp \
  --group-id sites --connection-name webapp-conn
```

---
## Summary

| Implementation | When to use |
|---------------|-------------|
| **Service Endpoint** | Simple, single-VNet, free. No on-prem access. |
| **Private Endpoint** | Zero-trust default. Private IP in your VNet. Needs DNS config. |
| **Private Link Service** | You're exposing your own service to external consumers. |
| **App Service VNet integration** | Outbound: VNet integration. Inbound: Private Endpoint. |
| **DNS zones** | `privatelink.<service>.core.windows.net` — link to VNet. |

**Next**: [Notebook 3 — Public Access and Firewalls](03_public_access_and_firewalls.ipynb)